# VidCap — Phase 0 + Stage B

**Before running:** Session options (right panel) → Accelerator `GPU T4 x2`, Internet **On**.

Full walkthrough: `KAGGLE.md` in the repo.

In [ ]:
# Cell 1 — deps. torch/transformers are preinstalled; SigLIP needs sentencepiece.
!pip install -q sentencepiece

In [ ]:
# Cell 2 — get the code.
GITHUB_USER = "Sampath7099"

!rm -rf /kaggle/working/VidCap
!git clone -q https://github.com/{GITHUB_USER}/VidCap.git /kaggle/working/VidCap
%cd /kaggle/working/VidCap

In [ ]:
# Cell 3a — fetch MSR-VTT (~6.55 GB). Skips if already present.
!test -d /kaggle/working/data || wget -q --show-progress \
    https://www.robots.ox.ac.uk/~maxbain/frozen-in-time/data/MSRVTT.zip \
    -O /kaggle/working/MSRVTT.zip

In [ ]:
# Cell 3b — unpack, normalise the path, verify. A truncated download looks fine until
# caching silently finds nothing, so the asserts are the point of this cell.
import glob, json, os

if not os.path.isdir("/kaggle/working/data/msrvtt"):
    !unzip -q /kaggle/working/MSRVTT.zip -d /kaggle/working/data
    # config.DATA expects <root>/msrvtt; the zip unpacks as MSRVTT.
    !mv /kaggle/working/data/MSRVTT /kaggle/working/data/msrvtt
    !rm -f /kaggle/working/MSRVTT.zip

# Every script resolves paths from this; set it once instead of --root everywhere.
os.environ["VIDCAP_DATA"] = "/kaggle/working/data"

root = "/kaggle/working/data/msrvtt"
n = len(glob.glob(f"{root}/**/*.mp4", recursive=True))
anns = [p for p in glob.glob(f"{root}/**/*.json", recursive=True)
        if "msr_vtt" in os.path.basename(p).lower()
        or "videodatainfo" in os.path.basename(p).lower()]
print(f"videos: {n}\nannotations: {anns}")
assert n > 9000, f"only {n} videos — zip is incomplete, rm -rf {root} and redo 3a"
assert anns, "no annotation JSON found"

# Schema varies by mirror; print it so a parse failure downstream has an obvious cause.
d = json.load(open(anns[0]))
print("top-level keys:", sorted(d.keys()))
for k in ("videos", "sentences", "annotations"):
    if isinstance(d.get(k), list) and d[k]:
        print(f"  {k}[0]: {d[k][0]}")
!df -h /kaggle/working | tail -1

In [ ]:
# Cell 4 — GATE: both frozen backbones load on GPU (~2 min, downloads ~6 GB).
# Expect 'vision ok on cuda dim=1152' and a Qwen line. If this fails, stop here.
!python -m scripts.smoke_test

In [ ]:
# Cell 4b — GATE: the LoRA/connector checks that can't run on a CPU laptop.
# VIDCAP_HEAVY=1 adds the real-config (Qwen fp32) overfit.
!VIDCAP_HEAVY=1 ./run_tests.sh

In [ ]:
# Cell 5 — cache 500 clips FIRST. ~10-20 min. Re-runnable: skips what's done.
# Debug on 500, not 10000 — GPU quota is ~30 h/week. Path comes from VIDCAP_DATA (Cell 3b).
!python -m scripts.build_cache msrvtt --limit 500

In [ ]:
# Cell 6 — first real training run. Loss should fall from ~8 and keep dropping.
!python -m scripts.train --stage B --limit 500 --epochs 3 --bs 16

In [ ]:
# Cell 7 — the blind control: identical training, video prefix zeroed.
!python -m scripts.train --stage B --limit 500 --epochs 3 --bs 16 --blind --name blind

In [ ]:
# Cell 8 — THE GATE. Sighted must clearly beat blind. If it doesn't, the model is
# ignoring the video and frame selection provably cannot matter. Stop and debug.
!python -m scripts.evaluate --ckpt stageB --ckpt-blind blind --limit 200 --budgets 8

In [ ]:
# Cell 9 — keep the embedding cache. /kaggle/working is wiped at session end and
# rebuilding costs GPU quota. Save Version, or download + re-upload as a Dataset.
!cd /kaggle/working && zip -qr cache.zip cache && ls -lh cache.zip